In [1]:
from datasets import load_dataset
import scipy as sp

# Load dataset

ds = load_dataset("ngwgsang/vietnamese-raw-sent", split="train")    

# Chỉ lấy 1 cột
texts = ds["text"]

# Giờ texts là list kiểu ['text1', 'text2', ...]


/usr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import re

def contains_chinese(text):
    return re.search(r'[\u4e00-\u9fff]', text) is not None

print(contains_chinese("hello"))             # False
print(contains_chinese("你好 thế giới"))       # True


False
True


In [1]:
import re
import json
import unicodedata

# def contains_chinese(text):
#     return re.search(r'[\u4e00-\u9fff]', text) is not None


def is_json_balanced(text):
    brackets = {'{': 0, '}': 0, '[': 0, ']': 0}
    for c in text:
        if c in brackets:
            brackets[c] += 1
    # Kiểm tra cân bằng
    return brackets['{'] == brackets['}'] and brackets['['] == brackets[']'], brackets


def remove_trailing_commas(json_text):
    # Xóa dấu phẩy cuối cùng trước dấu đóng } hoặc ]
    cleaned = re.sub(r',\s*(\}|\])', r'\1', json_text)
    return cleaned


def clean_json_text(text):
    # Xoá dấu , cuối cùng trước }
    text = re.sub(r',\s*(\}|\])', r'\1', text)

    # Optional: tìm các dòng chứa dấu " bên trong value mà không escape
    lines = text.splitlines()
    suspicious_lines = []

    for i, line in enumerate(lines):
        # Chỉ xét các dòng có pattern key-value
        if re.search(r'"\s*:\s*".*".*"', line):  # naive check
            suspicious_lines.append((i+1, line.strip()))

    return text, suspicious_lines



def clean_json_format(json_text):
    # 1. Xóa dấu `,` cuối cùng trước } hoặc ]
    cleaned = re.sub(r',\s*(\}|\])', r'\1', json_text)

    # 2. Thêm dấu `,` giữa } và { hoặc ] và [
    cleaned = re.sub(r'(\}|\])\s*\n\s*(\{|\[)', r'\1,\n\2', cleaned)

    # 3. Xóa dấu `,` mở đầu list hoặc sau [
    cleaned = re.sub(r'\[\s*,', '[', cleaned)

    # 4. Xóa dấu , liên tục dư
    cleaned = re.sub(r',\s*,+', ',', cleaned)

    return cleaned


def contains_chinese(text: str) -> bool:
    """Check nếu chuỗi chứa ký tự Trung Quốc"""
    return bool(re.search(r'[\u4e00-\u9fff]', text))


def filter_json_list(data: list) -> list:
    """In và loại bỏ các object có chứa ký tự Hán trong bất kỳ value nào"""
    cleaned = []

    for item in data:
        has_chinese = False
        for value in item.values():
            if isinstance(value, str) and contains_chinese(value):
                has_chinese = True
                break

        if has_chinese:
            print("🈸 Phát hiện object có chữ Tàu:", item)
        else:
            cleaned.append(item)

    return cleaned


def sanitize_json_text(raw_text: str) -> str:
    # 3. Thay thế dấu " trong value → '
    #    Naive: thay mọi dấu " bên trong value bằng '
    def fix_quotes_in_values(match):
        text = match.group(0)
        key, value = text.split(":", 1)
        # xử lý phần value: xoá dấu " và * bên trong, giữ nguyên ngoài
        value = value.strip()
        if value.startswith('"') and value.endswith('"'):
            inner = value[1:-1].replace('"', "'").replace("*", "'")
            fixed_value = f'"{inner}"'
        else:
            fixed_value = value
        return f"{key}: {fixed_value}"

    # xử lý các cặp key-value kiểu "key": "value"
    raw_text = re.sub(r'"[^"]*"\s*:\s*"[^"]*"', fix_quotes_in_values, raw_text)

def escape_inner_quotes_hehe(text):
    # Match toàn bộ chuỗi trong dấu ngoặc kép ngoài cùng
    pattern = r'^"(.*)"$'
    match = re.match(pattern, text)
    if not match:
        return text  # Nếu không đúng định dạng, trả về nguyên bản
    
    inner_text = match.group(1)
    # Thay thế tất cả dấu " bên trong thành \"
    escaped_inner = inner_text.replace('"', r'\"')
    
    # Ghép lại dấu ngoặc kép ngoài cùng
    return f'"{escaped_inner}"'


def hand(text: str) -> str:
    # Xử lý dấu ngoặc kép bên trong value
    res = []
    
    for line in text.split('\n'):
        if ":" in line:
            key, val = line.split(':', 1)
            if val.endswith(','):
                val = val.strip()[:-1] if val.endswith(',') else val.strip()
                val_handed = escape_inner_quotes_hehe(val) + ','
            else:
                val_handed = escape_inner_quotes_hehe(val.strip())
                
            res.append(f'{key.strip()}: {val_handed}')
        else:
            res.append(line)

    return '\n'.join(res)


def replace_custom_patterns(text):
    return re.sub(r'\*\*|\*o|\*n', lambda m: {
        '**': 'éo',
        '*o': 'éo',
        '*n': 'ồn',
        '\\*o': 'éo',
    }[m.group()], text)

def remove_before_but_keep_asterisk(text):

    return re.sub(r'(.)\*', r'*', text)



def contains_chinese(text: str) -> bool:
    """Kiểm tra nếu chuỗi có ký tự tiếng Trung (CJK Unified Ideographs)"""
    return bool(re.search(r'[\u4e00-\u9fff]', text))


def print_sources_with_chinese(json_path: str, output: str):
    """
    Đọc file JSON, in ra giá trị của key "source"
    nếu bất kỳ value nào trong object đó chứa chữ Trung Quốc.
    """
    c_senteces = []
    
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data:
        for val in item.values():
            if isinstance(val, str) and contains_chinese(val):
                c_senteces.append(item.get("source", "[NO SOURCE FOUND]"))
                break  # chỉ cần 1 value dính là in source rồi next luôn

    with open(output, "a", encoding="utf-8") as f:
        f.write("\n" + json_path + "\n")
        for source in c_senteces:
            f.write(source + "\n")

    return c_senteces


import json

def print_sources_with_chinese_new(json_path: str, output: str):
    """
    Lọc các item trong file JSON:
    - Bỏ những item mà chỉ 'clm' chứa chữ Trung Quốc.
    - In ra source của các item còn lại nếu có chữ Tàu nhưng 'clm' không bị.
    - Ghi lại file JSON sau khi lọc.
    """
    c_sentences = []

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    filtered_data = []

    for item in data:
        values = list(item.values())
        clm_val = item.get("4", "")

        # Trường hợp toàn bộ value đều có chữ Tàu → giữ nguyên
        if all(isinstance(v, str) and contains_chinese(v) for v in values):
            filtered_data.append(item)
            continue

        # Nếu chỉ clm có chữ Tàu, còn lại không → loại bỏ
        if (
            isinstance(clm_val, str) and contains_chinese(clm_val) and
            all(not contains_chinese(v) for k, v in item.items() if k != "4" and isinstance(v, str))
        ):
            continue  # Bỏ mẹ nó đi

        # Còn lại thì giữ và ghi source nếu có chữ Tàu
        if any(isinstance(v, str) and contains_chinese(v) for k, v in item.items()):
            c_sentences.append(item.get("source", "[NO SOURCE FOUND]"))

        filtered_data.append(item)

    # Ghi lại file JSON đã lọc
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=2)

    # Ghi source vào output
    with open(output, "a", encoding="utf-8") as f:
        f.write("\n" + json_path + "\n")
        for source in c_sentences:
            f.write(source + "\n")

    return c_sentences



def strip_vietnamese_diacritics(text: str) -> str:
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    return text

def normalize_text(text: str, remove_diacritics=False) -> str:
    text = text.strip()
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'\.{2,}$', '', text)
    text = re.sub(r'\.$', '', text)
    if remove_diacritics:
        text = strip_vietnamese_diacritics(text)
    if text.endswith('?'):
        text = text[:-1]
    return text

def find_missing_sources(txt_path: str, json_path: str, output_path: str):
    # 1. Đọc tất cả câu từ .txt và normalize
    with open(txt_path, "r", encoding="utf-8") as f:
        all_sentences = [normalize_text(line) for line in f if line.strip()]

    # 2. Đọc source trong .json và normalize
    with open(json_path, "r", encoding="utf-8") as f:
        json_data = json.load(f)
        json_sources = set(normalize_text(item.get("source", ""), True) for item in json_data)

    # 3. Tìm câu trong .txt mà không có trong .json
    missing = [s for s in all_sentences if normalize_text(s, True) not in json_sources]

    # 4. Ghi ra file
    with open(output_path, "a", encoding="utf-8") as f:
        f.write("\n" + json_path + "\n")
        for sentence in missing:
            f.write(sentence + "\n")

    print(f"✅ Tìm thấy {len(missing)} câu thiếu (sau chuẩn hoá). Ghi vào: {output_path}")



In [36]:
x = 255
with open(f'chunks_out/chunk_{x:04d}.json', 'r') as f:
    text = f.read()
    
    
t1 = hand(text)
t2 = clean_json_format(t1)
t3 = remove_before_but_keep_asterisk(t2)
t4 = replace_custom_patterns(t3)
# print(t)


with open(f'chunks_out/chunk_{x:04d}.json', 'w') as f:
    f.write(clean_json_format(text))

In [ ]:
with open('test_json/dirty.json', 'r') as f:
    text = f.read()
    
    
t1 = hand(text)
t2 = clean_json_format(t1)
t3 = remove_before_but_keep_asterisk(t2)
t4 = replace_custom_patterns(t3)
# print(t)



with open('test_json/cleaned_4.json', 'w') as f:
    f.write(t4)

In [140]:
print_sources_with_chinese('chunks_out/chunk_0145.json', 
                           'mismatch/chinese.txt')

[]

In [104]:
for i in range(180, 185):

    print_sources_with_chinese_new(
        f'chunks_out/chunk_{i:04d}.json', 
        'mismatch/chinese.txt')
    print(i)

180
181
182
183
184


In [ ]:
a = 140
find_missing_sources(
    f'chunks/chunk_{a:04d}.txt', 
    f'chunks_out/chunk_{a:04d}.json', 
    'mismatch/missing_sources.txt'
)

✅ Tìm thấy 8 câu thiếu (sau chuẩn hoá). Ghi vào: mismatch/missing_sources.txt


In [46]:
import re

text = '''

"cái "quần què" gì vậy" và "đây là "test case" số hai"

'''

def add_backslashes(match):
    inner = match.group(1)
    return f'"{match.group(0).split(inner)[0]}\\{inner}\\{match.group(0).split(inner)[-1]}"'

# hoặc ngắn gọn hơn:
new_text = re.sub(r'"([^"]*?)"([^"]*?)"([^"]*?)"', lambda m: f'"{m.group(1)}\\"{m.group(2)}\\"{m.group(3)}"', text)


def escape_inner_quotes(text: str) -> str:
    def replacer(match):
        before = match.group(1)
        inner = match.group(2)
        after = match.group(3)
        return f'"{before}\\{inner}\\{after}"'
    
    pattern = r'"([^"]*?)"([^"]*?)"([^"]*?)"'
    return re.sub(pattern, replacer, text)



def escape_inner_quotes_after_colon(text: str) -> str:
    def process_line(line: str) -> str:
        parts = line.split(':', 1)
        if len(parts) != 2:
            return line  # Bỏ qua dòng không hợp lệ
        
        key, value = parts[0], parts[1]

        # Chỉ escape dấu " nằm trong value
        def add_backslashes(m):
            return f'\\"{m.group(1)}\\"'

        # Tìm "inner" quote trong phần value
        value_escaped = re.sub(r'"([^"]*?)"', add_backslashes, value)
        return f'{key}: {value_escaped}'

    lines = text.splitlines()
    processed = [process_line(line) for line in lines]
    return '\n'.join(processed)

In [52]:
import re

def escape_inner_quotes(text):
    # Match toàn bộ chuỗi trong dấu ngoặc kép ngoài cùng
    pattern = r'^"(.*)"$'
    match = re.match(pattern, text)
    if not match:
        return text  # Nếu không đúng định dạng, trả về nguyên bản
    
    inner_text = match.group(1)
    # Thay thế tất cả dấu " bên trong thành \"
    escaped_inner = inner_text.replace('"', r'\"')
    
    # Ghép lại dấu ngoặc kép ngoài cùng
    return f'"{escaped_inner}"'

# Test
input_text = '"cái "quần què" gì vậy thằng "mát" kia?"'
output_text = escape_inner_quotes(input_text)
print(output_text)


"cái \"quần què\" gì vậy thằng \"mát\" kia?"


In [40]:
with open('test_json/dirty.json', 'r') as f:
    text = f.read()
    
import json


print(text)
print("----")
print(escape_inner_quotes(text))
print("----")
print(escape_inner_quotes_after_colon(text))

with open('test_json/cleaned.json', 'w') as f:
    f.write(escape_inner_quotes_after_colon(text))



[
  {
    "1": "Ê thử que rồi mà "dì" chưa tới *nghi* rồi",
    "2": "Đèn đỏ vẫn chưa thấy",
    "3": "OK "hả cái gì cơ", à hiểu rồi"
  },
  {
    "1": "Thử que xong thấy đ\*o có thai, nhưng kinh vẫn chưa tới"
  }
]
----
[
  {
    "1\: \Ê thử que rồi mà "dì" chưa tới *nghi* rồi\,
    \2": "Đèn đỏ vẫn chưa thấy\,
    \3": "OK \hả cái gì cơ\, à hiểu rồi"
  },
  {
    "1\: \Thử que xong thấy đ\*o có thai, nhưng kinh vẫn chưa tới"
  }
]
----
[
  {
    "1":  \"Ê thử que rồi mà \"dì\" chưa tới *nghi* rồi\",
    "2":  \"Đèn đỏ vẫn chưa thấy\",
    "3":  \"OK \"hả cái gì cơ\", à hiểu rồi\"
  },
  {
    "1":  \"Thử que xong thấy đ\*o có thai, nhưng kinh vẫn chưa tới\"
  }
]


In [30]:
import re

def escape_nested_quotes(text: str) -> str:
    def replacer(match):
        key = match.group(1)
        value = match.group(2)

        # Nếu value được bọc bằng dấu ", xử lý phần giữa thôi
        if value.startswith('"') and value.endswith('"'):
            inner = value[1:-1]
            escaped_inner = re.sub(r'(?<!\\)"', r'\"', inner)
            return f'{key}: "{escaped_inner}"'
        return match.group(0)

    return re.sub(r'^(.*?:)\s*(".*?")', replacer, text, flags=re.MULTILINE)


In [77]:
sample = '''
"cái "đầu buoi" gì ku?": "cái "quần què" gì vậy thằng "mát" kia? " 
"bình thường": "không sao đâu"  
"nói ": "câu "này" có vấn đề"   
'''

for line in sample.split('\n'):
    val = line.split(':')[1].strip() if ':' in line else line
    
    print(val)
    print(escape_inner_quotes_hehe(val))




"cái "quần què" gì vậy thằng "mát" kia? "
"cái \"quần què\" gì vậy thằng \"mát\" kia? "
"không sao đâu"
"không sao đâu"
"câu "này" có vấn đề"
"câu \"này\" có vấn đề"




In [4]:
import json

def hand(text: str) -> str:
    # Xử lý dấu ngoặc kép bên trong value
    res = []
    
    for line in text.split('\n'):
        if ":" in line:
            key, val = line.split(':', 1)
            if val.endswith(','):
                val = val.strip()[:-1] if val.endswith(',') else val.strip()
                val_handed = escape_inner_quotes_hehe(val) + ','
            else:
                val_handed = escape_inner_quotes_hehe(val.strip())
                
            res.append(f'{key.strip()}: {val_handed}')
        else:
            res.append(line)

    return '\n'.join(res)




with open('test_json/dirty.json', 'r') as f:
    text = f.read()


print(hand(text))

json_text = json.loads(hand(text))

NameError: name 'escape_inner_quotes_hehe' is not defined